<a href="https://colab.research.google.com/github/rlguss1234-web/lotto-ai-practice/blob/main/%EB%A1%9C%EB%98%90%EB%B6%84%EC%84%9D%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import random
from collections import Counter

def generate_optimized_combinations(lotto_dict, latest_draw_no, total_weeks=52, num_combinations=5):
    # 1. 최근 기간 데이터 수집
    history = []
    target_draws = list(range(latest_draw_no - total_weeks + 1, latest_draw_no + 1))
    for d in target_draws:
        if d in lotto_dict:
            history.append(lotto_dict[d])

    if not history:
        print("분석할 데이터가 없습니다.")
        return

    # 2. 최다 및 최저 출현 번호 분석
    all_numbers = [num for draw in history for num in draw]
    counts = Counter(all_numbers)
    full_freq = {n: counts.get(n, 0) for n in range(1, 46)}
    sorted_freq = sorted(full_freq.items(), key=lambda x: x[1], reverse=True)

    # 상위 15개 (최다), 하위 15개 (최저) 번호 풀 구성
    frequent_pool = [n for n, c in sorted_freq[:15]]
    infrequent_pool = [n for n, c in sorted_freq[-15:]]

    # 균형잡힌 가중치 번호 풀 (최다 출현 번호에 더 높은 선택 가중치 부여)
    combined_pool = []
    for n in range(1, 46):
        weight = full_freq[n] + 1  # 0번 나온 번호도 최소한의 기회를 갖도록 +1
        combined_pool.extend([n] * weight)

    generated = []
    attempts = 0

    while len(generated) < num_combinations and attempts < 10000:
        attempts += 1
        # 최다 출현 번호풀에서 3~4개, 최저 출현 번호풀에서 1~2개, 전체 풀에서 나머지를 샘플링하여 균형 조합 생성
        sample_freq = random.sample(frequent_pool, random.randint(3, 4))
        sample_infreq = random.sample(infrequent_pool, random.randint(1, 2))
        remaining_needed = 6 - (len(sample_freq) + len(sample_infreq))

        candidate = set(sample_freq + sample_infreq)
        while len(candidate) < 6:
            candidate.add(random.choice(combined_pool))

        candidate_list = sorted(list(candidate))

        # 조건 1: 총합 범위가 120 ~ 180 사이인지 검사
        total_sum = sum(candidate_list)
        if not (120 <= total_sum <= 180):
            continue

        # 조건 4: 홀수/짝수 비율 검사 (3:3, 4:2, 2:4)
        odds = sum(1 for n in candidate_list if n % 2 != 0)
        evens = 6 - odds
        if (odds, evens) not in [(3, 3), (4, 2), (2, 4)]:
            continue

        if candidate_list not in generated:
            generated.append(candidate_list)

    # 결과 출력
    print(f"=== 최근 {total_weeks}주 데이터 기반 추천 조합 (조건 만족) ===\n")
    for i, combo in enumerate(generated, 1):
        total_sum = sum(combo)
        odds = sum(1 for n in combo if n % 2 != 0)
        evens = 6 - odds
        print(f"조합 {i}: {combo}  [총합: {total_sum}] [홀짝 비율: {odds}:{evens}]")

# 조합 생성기 실행
generate_optimized_combinations(lotto_data, latest_draw, total_weeks=52, num_combinations=5)

=== 최근 52주 데이터 기반 추천 조합 (조건 만족) ===

조합 1: [2, 18, 22, 29, 31, 34]  [총합: 136] [홀짝 비율: 2:4]
조합 2: [1, 2, 27, 29, 32, 37]  [총합: 128] [홀짝 비율: 4:2]
조합 3: [1, 2, 16, 27, 31, 43]  [총합: 120] [홀짝 비율: 4:2]
조합 4: [15, 22, 29, 32, 37, 38]  [총합: 173] [홀짝 비율: 3:3]
조합 5: [1, 18, 22, 39, 43, 44]  [총합: 167] [홀짝 비율: 3:3]
